In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Phase1 v2 CPU：尚待一次具体授权
固定公共C取全部32时刻，允许稳定旋转/反射/混合；两内容各共同15/16且12臂总质量硬门。旧raw对角指标只诊断。
本笔记本会真实修改像素并编码12视频，只在批准固定2输入/六臂/v2参数后执行。无GPU。当前媒体单元未运行。

In [ ]:
from pathlib import Path
import zipfile,sys,subprocess,json
ZIP=Path('/content/drive/MyDrive/Video-WM/public_statistic_phase1_source.zip')
ROOT=Path('/content/public_statistic_phase1')
if ROOT.exists(): raise FileExistsError(ROOT)
with zipfile.ZipFile(ZIP) as z:
    if any(Path(n).is_absolute() or '..' in Path(n).parts for n in z.namelist()): raise ValueError('unsafe archive path')
    z.extractall(ROOT)
subprocess.run([sys.executable,'-m','pip','install','numpy'],check=True)
for executable in ('ffmpeg','ffprobe'): subprocess.run([executable,'-version'],check=True,capture_output=True)


In [ ]:
P50=Path('/content/drive/MyDrive/Video-WM/p50_motion_audit/saved.mp4')
JUMP=Path('/content/drive/MyDrive/Video-WM/datasets/UCF101/JumpingJack/v_JumpingJack_g01_c01.avi')
for p in (P50,JUMP):
    if not p.is_file(): raise FileNotFoundError(p)
c=json.loads((ROOT/'configs/public_luma_phase1.json').read_text())
c['inputs'][0]['path']=str(P50); c['inputs'][1]['path']=str(JUMP)
CONFIG=ROOT/'local_paths.json'
with CONFIG.open('x') as f: json.dump(c,f,indent=2)
OUTPUT=Path('/content/public_statistic_phase1_output')


In [ ]:
import os
if OUTPUT.exists() or OUTPUT.with_suffix('.log').exists() or OUTPUT.with_suffix('.exit.json').exists(): raise FileExistsError(OUTPUT)
env=dict(os.environ,PYTHONPATH=str(ROOT),CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='1')
subprocess.run([sys.executable,str(ROOT/'experiments/public_statistic/run_phase1.py'),'--config',str(CONFIG),'--output',str(OUTPUT)],cwd=ROOT,env=env,check=True)


结果保留在本地/content，不自动写回Drive。读取result.json中全部192行及每臂原片；视觉质量必须另外人工审核，不能将数值门当科学或感知通过。